# Exp-2 — HyDE + Issue-Spot Enumeration (Qwen2.5-7B)

**Why this exp.** Exp-1 showed BGE-M3 alone reaches only 21.5% stat_recall@500 on val — well below the 40% gate. Per-query analysis showed the gap is *conceptual*: statute articles are abstract principles (e.g. Art. 397 OR = mandate revocability) whose retrieval from a factual scenario requires legal reasoning, not surface similarity. This exp tests whether an LLM-in-the-loop bridges that gap.

**Two prompts per query, both run on Qwen/Qwen2.5-7B-Instruct (local, no API key):**

1. **HyDE** — write 3 hypothetical German statute-style passages that govern the scenario.
2. **Issue-spotting enumeration** — list 10–15 likely Swiss article citations (e.g. `Art. 397 OR`, `Art. 2 ZGB`) with a one-line reason each. Output is concatenated into a pseudo-query string.

Each expansion is encoded with BGE-M3 and searched against the cached `laws_bgem3.npy`. Results fused via **Reciprocal Rank Fusion** with the raw EN-query dense retrieval from Exp-1.

**Gate**: stat_recall@500 must reach ≥ 0.40 on val. Otherwise, we keep layering (BM25, reranker) but the plan calls for revisiting the overall strategy.

**Inputs on Drive (`MyDrive/swiss_law/`):**
- `laws_de.csv`
- `val.csv`
- `val_translated_de.pkl`
- `artifacts/laws_bgem3.npy` (from Exp-1)

**Outputs:**
- `artifacts/exp_A2_expansions.json` — raw LLM outputs per query (for debugging)
- `artifacts/exp_A2_report.json` — recall table, per-query hits
- `artifacts/query_vecs.npz` — encoded query/expansion embeddings, reused by Exp-3/4

In [ ]:
# --- Cell 1. Install deps (Qwen via vLLM for fast batched inference) ---
!pip install -q FlagEmbedding==1.3.5 vllm pandas numpy transformers>=4.45.0

import os
print("Restarting runtime...")
os.kill(os.getpid(), 9)

In [2]:
# --- Cell 2. Mount Drive & paths ---
from google.colab import drive
drive.mount('/content/drive')

import json, pickle, re, time
from pathlib import Path
import numpy as np
import pandas as pd

ROOT = Path('/content/drive/MyDrive/swiss_law/data')
ART  = ROOT / 'artifacts'
ART.mkdir(parents=True, exist_ok=True)
assert (ART / 'laws_bgem3.npy').exists(), 'run Exp-1 first'

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


In [3]:
# --- Cell 3. Load val + corpus metadata ---
val  = pd.read_csv(ROOT / 'val.csv')
laws = pd.read_csv(ROOT / 'laws_de.csv')
with open(ROOT / 'val_translated_de.pkl', 'rb') as f:
    val_de = pickle.load(f)
cits = laws['citation'].tolist()
doc_emb = np.load(ART / 'laws_bgem3.npy').astype(np.float32)
print('corpus:', doc_emb.shape, '| val:', len(val))

corpus: (175933, 1024) | val: 10


In [ ]:
!pip install --upgrade vllm transformers accelerate


In [1]:
# --- Cell 4. Load Qwen2.5-7B via vLLM ---
import os
import sys

os.environ["VLLM_NO_USAGE_STATS"] = "1"

# Workaround for io.UnsupportedOperation: fileno in Colab
# Override explicitly because Colab's stream has the method but it raises an error
sys.stdout.fileno = lambda: 1
sys.stderr.fileno = lambda: 2

from vllm import LLM, SamplingParams

llm = LLM(model='Qwen/Qwen2.5-7B-Instruct',
          dtype='bfloat16',
          gpu_memory_utilization=0.80,
          trust_remote_code=True)

def chat(messages, **kw):
    # vllm chat template
    params = SamplingParams(temperature=kw.get('temperature', 0.2),
                            top_p=0.9,
                            max_tokens=kw.get('max_tokens', 800))
    out = llm.chat(messages, sampling_params=params, use_tqdm=False)
    return out[0].outputs[0].text.strip()

INFO 04-21 13:31:00 [utils.py:233] non-default args: {'trust_remote_code': True, 'dtype': 'bfloat16', 'gpu_memory_utilization': 0.8, 'disable_log_stats': True, 'model': 'Qwen/Qwen2.5-7B-Instruct'}
INFO 04-21 13:31:02 [model.py:549] Resolved architecture: Qwen2ForCausalLM
INFO 04-21 13:31:02 [model.py:1678] Using max model len 32768
INFO 04-21 13:31:02 [scheduler.py:238] Chunked prefill is enabled with max_num_batched_tokens=16384.
INFO 04-21 13:31:02 [vllm.py:790] Asynchronous scheduling is enabled.
(EngineCore pid=12874) INFO 04-21 13:31:03 [core.py:105] Initializing a V1 LLM engine (v0.19.1) with config: model='Qwen/Qwen2.5-7B-Instruct', speculative_config=None, tokenizer='Qwen/Qwen2.5-7B-Instruct', skip_tokenizer_init=False, tokenizer_mode=auto, revision=None, tokenizer_revision=None, trust_remote_code=True, dtype=torch.bfloat16, max_seq_len=32768, download_dir=None, load_format=auto, tensor_parallel_size=1, pipeline_parallel_size=1, data_parallel_size=1, decode_context_parallel_siz

(EngineCore pid=12874) <frozen importlib._bootstrap_external>:1301: FutureWarning: The cuda.cudart module is deprecated and will be removed in a future release, please switch to use the cuda.bindings.runtime module instead.
(EngineCore pid=12874) <frozen importlib._bootstrap_external>:1301: FutureWarning: The cuda.nvrtc module is deprecated and will be removed in a future release, please switch to use the cuda.bindings.nvrtc module instead.


Loading safetensors checkpoint shards:   0% Completed | 0/4 [00:00<?, ?it/s]


(EngineCore pid=12874) INFO 04-21 13:31:07 [default_loader.py:384] Loading weights took 1.62 seconds
(EngineCore pid=12874) INFO 04-21 13:31:08 [gpu_model_runner.py:4820] Model loading took 14.25 GiB memory and 2.409498 seconds
(EngineCore pid=12874) INFO 04-21 13:31:09 [backends.py:1051] Using cache directory: /root/.cache/vllm/torch_compile_cache/4278c45112/rank_0_0/backbone for vLLM's torch.compile
(EngineCore pid=12874) INFO 04-21 13:31:09 [backends.py:1111] Dynamo bytecode transform time: 1.06 s
(EngineCore pid=12874) INFO 04-21 13:31:10 [backends.py:285] Directly load the compiled graph(s) for compile range (1, 16384) from the cache, took 0.676 s
(EngineCore pid=12874) INFO 04-21 13:31:10 [decorators.py:305] Directly load AOT compilation from path /root/.cache/vllm/torch_compile_cache/torch_aot_compile/783162e3741bf2ba709aa622a204bae586fa062a772b009ea0a1dcb8b3cfa89e/rank_0_0/model
(EngineCore pid=12874) INFO 04-21 13:31:10 [monitor.py:48] torch.compile took 1.86 s in total
(Engin

(EngineCore pid=12874) 2026-04-21 13:31:17,055 - INFO - autotuner.py:262 - flashinfer.jit: [Autotuner]: Autotuning process starts ...
(EngineCore pid=12874) 2026-04-21 13:31:17,060 - INFO - autotuner.py:268 - flashinfer.jit: [Autotuner]: Autotuning process ends
Capturing CUDA graphs (mixed prefill-decode, PIECEWISE): 100%|██████████| 51/51 [00:01<00:00, 41.92it/s]
Capturing CUDA graphs (decode, FULL): 100%|██████████| 51/51 [00:01<00:00, 46.39it/s]


(EngineCore pid=12874) INFO 04-21 13:31:20 [gpu_model_runner.py:6046] Graph capturing finished in 3 secs, took 0.38 GiB
(EngineCore pid=12874) INFO 04-21 13:31:20 [gpu_worker.py:597] CUDA graph pool memory: 0.38 GiB (actual), 0.41 GiB (estimated), difference: 0.04 GiB (10.4%).
(EngineCore pid=12874) INFO 04-21 13:31:20 [core.py:283] init engine (profile, create kv cache, warmup model) took 12.27 seconds


In [2]:
# --- Cell 5. Prompts ---
HYDE_SYS = (
    'Du bist Schweizer Jurist. Schreibe deutsche Texte im Stil Schweizer Bundesgesetze.'
)
HYDE_USER = (
    'Gegeben ist folgender Sachverhalt (auf Englisch):\n\n{q}\n\n'
    'Schreibe DREI hypothetische deutsche Paragraphen im Stil Schweizer Bundesgesetze '
    '(OR, ZGB, StGB, StPO, ZPO, BGG, etc.), die diesen Sachverhalt rechtlich regeln. '
    'Jeder Paragraph 2-4 Sätze. Formal-juristischer Ton. '
    'Trenne die drei Paragraphen mit \"---\". Keine Überschriften, kein Vorspann, '
    'nur die drei Paragraphen.'
)

ENUM_SYS = (
    'You are a Swiss legal expert. You know the Swiss Federal Codes (OR, ZGB, StGB, StPO, '
    'ZPO, BGG, BV, IPRG, SchKG, DBG, StHG, etc.) and typical articles cited for common '
    'legal issues.'
)
ENUM_USER = (
    'Scenario:\n\n{q}\n\n'
    'List 15 Swiss federal law citations a Swiss lawyer would most likely consult to analyse '
    'this case. Use exact format \"Art. X Abs. Y ABBR\" or \"Art. X ABBR\" (e.g. \"Art. 397 '
    'Abs. 1 OR\", \"Art. 2 ZGB\"). One citation per line, followed by a 4-8 word reason '
    'separated by \" — \". No numbering, no headers, no extra commentary.'
)

def build_prompts(q_en):
    return (
        [{'role': 'system', 'content': HYDE_SYS},
         {'role': 'user',   'content': HYDE_USER.format(q=q_en)}],
        [{'role': 'system', 'content': ENUM_SYS},
         {'role': 'user',   'content': ENUM_USER.format(q=q_en)}],
    )

In [6]:
# --- Cell 6. Generate expansions ---
expansions = {}
for row in val.itertuples():
    q = row.query
    hyde_msgs, enum_msgs = build_prompts(q)
    hyde_txt = chat(hyde_msgs, temperature=0.3, max_tokens=700)
    enum_txt = chat(enum_msgs, temperature=0.1, max_tokens=600)
    expansions[row.query_id] = {
        'query_en': q,
        'query_de': val_de[row.query_id],
        'hyde':  hyde_txt,
        'enum':  enum_txt,
    }
    print(f'{row.query_id}: HyDE {len(hyde_txt)}c | Enum {len(enum_txt)}c')

with open(ART / 'exp_A2_expansions.json', 'w') as f:
    json.dump(expansions, f, ensure_ascii=False, indent=2)
print('saved expansions')

# peek
print('--- val_001 enum preview ---')
print(expansions['val_001']['enum'][:700])
print('--- val_001 hyde preview ---')
print(expansions['val_001']['hyde'][:700])

INFO 04-21 13:31:47 [hf.py:314] Detected the chat template content format to be 'string'. You can set `--chat-template-content-format` to override this.
val_001: HyDE 2291c | Enum 737c
val_002: HyDE 1875c | Enum 994c
val_003: HyDE 1923c | Enum 720c
val_004: HyDE 1425c | Enum 757c
val_005: HyDE 2176c | Enum 595c
val_006: HyDE 1761c | Enum 904c
val_007: HyDE 1833c | Enum 541c
val_008: HyDE 1763c | Enum 635c
val_009: HyDE 1601c | Enum 685c
val_010: HyDE 1559c | Enum 972c
saved expansions
--- val_001 enum preview ---
Art. 221 Abs. 1 lit. b StPO — Risk of collusion
Art. 221 Abs. 2 StPO — Proportionality
Art. 221 Abs. 3 StPO — Considerations for prolongation
Art. 22 StPO — Detention periods
Art. 221 Abs. 1 lit. a StPO — Risk of flight
Art. 221 Abs. 1 lit. c StPO — Risk of reoffending
Art. 221 Abs. 1 lit. d StPO — Risk of destroying evidence
Art. 221 Abs. 1 lit. e StPO — Risk of influencing witnesses
Art. 221 Abs. 1 lit. f StPO — Risk of tampering with evidence
Art. 221 Abs. 4 StPO — Review o

In [7]:
# --- Cell 7. Free the Qwen GPU memory before loading BGE-M3 ---
import gc, torch
del llm
gc.collect(); torch.cuda.empty_cache()
print('cleared')

cleared


In [1]:
# --- Cell 1. Install deps ---
!pip install -q FlagEmbedding pandas numpy transformers==4.44.2


In [4]:
# --- Cell 8. Encode queries + expansions with BGE-M3 ---
import json
import numpy as np
from FlagEmbedding import BGEM3FlagModel

# If the runtime was restarted, load the LLM expansions from disk
if 'expansions' not in locals():
    print("Loading expansions from disk...")
    with open(ART / 'exp_A2_expansions.json', 'r') as f:
        expansions = json.load(f)

enc = BGEM3FlagModel('BAAI/bge-m3', use_fp16=True)

def embed(texts, max_len=2048):
    return enc.encode(texts, batch_size=4, max_length=max_len,
                      return_dense=True, return_sparse=False,
                      return_colbert_vecs=False)['dense_vecs']

en_qs   = [expansions[r.query_id]['query_en'] for r in val.itertuples()]
de_qs   = [expansions[r.query_id]['query_de'] for r in val.itertuples()]
hyde_qs = [expansions[r.query_id]['hyde']     for r in val.itertuples()]
enum_qs = [expansions[r.query_id]['enum']     for r in val.itertuples()]

q_en   = embed(en_qs)
q_de   = embed(de_qs)
q_hyde = embed(hyde_qs)
q_enum = embed(enum_qs, max_len=1024)

np.savez(ART / 'query_vecs.npz',
         q_en=q_en, q_de=q_de, q_hyde=q_hyde, q_enum=q_enum,
         query_ids=np.array([r.query_id for r in val.itertuples()]))
print('query vecs:', {k: v.shape for k, v in [('en',q_en),('de',q_de),('hyde',q_hyde),('enum',q_enum)]})

Loading expansions from disk...


Fetching 30 files:   0%|          | 0/30 [00:00<?, ?it/s]

pre tokenize: 100%|██████████| 3/3 [00:00<00:00, 520.28it/s]
You're using a XLMRobertaTokenizerFast tokenizer. Please note that with a fast tokenizer, using the `__call__` method is faster than using a method to encode the text followed by a call to the `pad` method to get a padded encoding.
Inference Embeddings: 100%|██████████| 3/3 [00:00<00:00, 160.73it/s]

query vecs: {'en': (10, 1024), 'de': (10, 1024), 'hyde': (10, 1024), 'enum': (10, 1024)}


In [5]:
# --- Cell 9. Retrieval + RRF fusion ---
# For each query form, rank all 175k docs by cosine similarity; fuse via RRF (k=60).

def rank_all(q):
    sims = q @ doc_emb.T                  # (N_q, N_doc)
    return np.argsort(-sims, axis=1)      # full rank list per query

ranks = {
    'en':   rank_all(q_en),
    'de':   rank_all(q_de),
    'hyde': rank_all(q_hyde),
    'enum': rank_all(q_enum),
}

def rrf(rank_lists, k_rrf=60, topk=2000):
    # rank_lists: list of (N_q, N_doc) full rankings
    N_q, N_doc = rank_lists[0].shape
    scores = np.zeros((N_q, N_doc), dtype=np.float32)
    for rl in rank_lists:
        # rl[i, r] = doc_idx at rank r; we need position of each doc in rl
        pos = np.empty_like(rl)
        rows = np.arange(N_q)[:, None]
        pos[rows, rl] = np.arange(N_doc)[None, :]
        scores += 1.0 / (k_rrf + pos.astype(np.float32))
    idx = np.argpartition(-scores, topk - 1, axis=1)[:, :topk]
    rows = np.arange(N_q)[:, None]
    order = np.argsort(-scores[rows, idx], axis=1)
    return idx[rows, order]

def parse(s): return [c.strip() for c in str(s).split(';') if c.strip()]
def is_statute(c):
    return not (c.startswith('BGE ') or re.match(r'\d[A-Z]_', c) or re.match(r'[A-Z]\d[A-Z]_', c))

def eval_ranking(top_idx, label, ks=(50, 200, 500, 1000)):
    per_q = []
    for i, row in enumerate(val.itertuples()):
        gold_stat = {c for c in parse(row.gold_citations) if is_statute(c)}
        retrieved = [cits[j] for j in top_idx[i]]
        entry = {'query_id': row.query_id, 'n_gold_stat': len(gold_stat)}
        for k in ks:
            entry[f'stat_hit@{k}'] = len(gold_stat & set(retrieved[:k]))
        per_q.append(entry)
    agg = {f'stat_recall@{k}': sum(p[f'stat_hit@{k}'] for p in per_q)
                              / max(1, sum(p['n_gold_stat'] for p in per_q)) for k in ks}
    print(f'=== {label} ===')
    for k, v in agg.items():
        print(f'  {k} = {v:.3f}')
    return {'agg': agg, 'per_query': per_q}

# baselines + fused combinations
report = {}
report['en_only']      = eval_ranking(ranks['en'][:, :1000], 'EN only (Exp-1 baseline)')
report['hyde_only']    = eval_ranking(ranks['hyde'][:, :1000], 'HyDE only')
report['enum_only']    = eval_ranking(ranks['enum'][:, :1000], 'Enum only')
report['en+hyde']      = eval_ranking(rrf([ranks['en'], ranks['hyde']]), 'RRF(en, hyde)')
report['en+enum']      = eval_ranking(rrf([ranks['en'], ranks['enum']]), 'RRF(en, enum)')
report['hyde+enum']    = eval_ranking(rrf([ranks['hyde'], ranks['enum']]), 'RRF(hyde, enum)')
report['en+hyde+enum'] = eval_ranking(rrf([ranks['en'], ranks['hyde'], ranks['enum']]), 'RRF(en, hyde, enum)')
report['all4']         = eval_ranking(rrf([ranks['en'], ranks['de'], ranks['hyde'], ranks['enum']]), 'RRF(en, de, hyde, enum)')

=== EN only (Exp-1 baseline) ===
  stat_recall@50 = 0.074
  stat_recall@200 = 0.121
  stat_recall@500 = 0.208
  stat_recall@1000 = 0.302
=== HyDE only ===
  stat_recall@50 = 0.094
  stat_recall@200 = 0.161
  stat_recall@500 = 0.235
  stat_recall@1000 = 0.255
=== Enum only ===
  stat_recall@50 = 0.114
  stat_recall@200 = 0.161
  stat_recall@500 = 0.235
  stat_recall@1000 = 0.342
=== RRF(en, hyde) ===
  stat_recall@50 = 0.081
  stat_recall@200 = 0.161
  stat_recall@500 = 0.242
  stat_recall@1000 = 0.322
=== RRF(en, enum) ===
  stat_recall@50 = 0.107
  stat_recall@200 = 0.181
  stat_recall@500 = 0.275
  stat_recall@1000 = 0.383
=== RRF(hyde, enum) ===
  stat_recall@50 = 0.107
  stat_recall@200 = 0.195
  stat_recall@500 = 0.295
  stat_recall@1000 = 0.362
=== RRF(en, hyde, enum) ===
  stat_recall@50 = 0.107
  stat_recall@200 = 0.195
  stat_recall@500 = 0.275
  stat_recall@1000 = 0.369
=== RRF(en, de, hyde, enum) ===
  stat_recall@50 = 0.101
  stat_recall@200 = 0.181
  stat_recall@500 = 0.27

In [6]:
# --- Cell 10. Save report + verdict ---
report['meta'] = {'model_retriever': 'BAAI/bge-m3',
                  'model_llm': 'Qwen/Qwen2.5-7B-Instruct',
                  'n_queries': len(val),
                  'exp1_baseline_recall@500': 0.215,
                  'gate': 'stat_recall@500 >= 0.40'}
with open(ART / 'exp_A2_report.json', 'w') as f:
    json.dump(report, f, indent=2, default=str)

best_key = max((k for k in report if k != 'meta'),
               key=lambda k: report[k]['agg']['stat_recall@500'])
best_val = report[best_key]['agg']['stat_recall@500']
print(f"best variant: {best_key} -> stat_recall@500 = {best_val:.3f} "
      f"(Exp-1 was 0.215; gate 0.40 -> {'PASS' if best_val >= 0.40 else 'BELOW GATE'})")

best variant: hyde+enum -> stat_recall@500 = 0.295 (Exp-1 was 0.215; gate 0.40 -> BELOW GATE)
